In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv(r"C:\Users\Santosh Agasti\Downloads\heart.csv")

# Display first 5 rows
print(df.head())

# Dataset Information
print(df.info())

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 

In [3]:
print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows before removing duplicates: 1025
Rows after removing duplicates: 302


In [25]:
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)
df["chol"] = df["chol"].fillna(df["chol"].median())
df["trestbps"] = df["trestbps"].fillna(df["trestbps"].median())

# Saving the cleaned dataset:
df.to_csv("heart_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [26]:
X = df.drop("target", axis=1)
y = df["target"]

In [27]:
categorical_features = ["cp", "thal", "slope", "restecg"]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

In [28]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)




In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [30]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


In [31]:
print("\nLogistic Regression:")

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

log_acc = accuracy_score(y_test, log_pred)

print("Accuracy:", log_acc)
print("\nConfusion Matrix")
print(confusion_matrix(y_test, log_pred))
print("\nClassification Report")
print(classification_report(y_test, log_pred))


Logistic Regression:
Accuracy: 0.8524590163934426

Confusion Matrix
[[24  4]
 [ 5 28]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.86      0.84        28
           1       0.88      0.85      0.86        33

    accuracy                           0.85        61
   macro avg       0.85      0.85      0.85        61
weighted avg       0.85      0.85      0.85        61



In [32]:
print("\n K-Nearest Neighbors: ")

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)

knn_acc = accuracy_score(y_test, knn_pred)

print("Accuracy:", knn_acc)
print("\nConfusion Matrix")
print(confusion_matrix(y_test, knn_pred))
print("\nClassification Report")
print(classification_report(y_test, knn_pred))



 K-Nearest Neighbors: 
Accuracy: 0.819672131147541

Confusion Matrix
[[23  5]
 [ 6 27]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.82      0.81        28
           1       0.84      0.82      0.83        33

    accuracy                           0.82        61
   macro avg       0.82      0.82      0.82        61
weighted avg       0.82      0.82      0.82        61



In [33]:
print("\nDecision Tree :")

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

dt_acc = accuracy_score(y_test, dt_pred)

print("Accuracy:", dt_acc)
print("\nConfusion Matrix")
print(confusion_matrix(y_test, dt_pred))
print("\nClassification Report")
print(classification_report(y_test, dt_pred))



Decision Tree :
Accuracy: 0.7377049180327869

Confusion Matrix
[[19  9]
 [ 7 26]]

Classification Report
              precision    recall  f1-score   support

           0       0.73      0.68      0.70        28
           1       0.74      0.79      0.76        33

    accuracy                           0.74        61
   macro avg       0.74      0.73      0.73        61
weighted avg       0.74      0.74      0.74        61



In [23]:
from sklearn.metrics import f1_score
log_f1 = f1_score(y_test, log_pred)
knn_f1 = f1_score(y_test, knn_pred)
dt_f1 = f1_score(y_test, dt_pred)


comparison = pd.DataFrame({
    "Algorithm": [
        "Logistic Regression",
        "K-Nearest Neighbors",
        "Decision Tree"
    ],
    "Accuracy": [
        f"{log_acc*100:.0f}%",
        f"{knn_acc*100:.0f}%",
        f"{dt_acc*100:.0f}%"
    ],
    "F1 Score": [
        round(log_f1, 2),
        round(knn_f1, 2),
        round(dt_f1, 2)
    ]
})

print("\nMODEL COMPARISON:")
print(comparison)

best_accuracy = comparison['Accuracy'].max()

print("\nBEST PERFORMING ALGORITHM:")
print(comparison.loc[comparison['Accuracy'] == best_accuracy])


MODEL COMPARISON:
             Algorithm Accuracy  F1 Score
0  Logistic Regression      85%      0.86
1  K-Nearest Neighbors      82%      0.83
2        Decision Tree      74%      0.76

BEST PERFORMING ALGORITHM:
             Algorithm Accuracy  F1 Score
0  Logistic Regression      85%      0.86
